In [1]:
import itertools
import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.metrics import mean_squared_error

# Read data from CSV file (assumes tab-separated or adjust accordingly)
data = pd.read_csv('fuel.dat', sep='\t')  # Change sep=',' if comma-separated

# Function to fit linear regression model and calculate metrics
def fit_model_and_calculate_metrics(features, target, data):
    X = sm.add_constant(data[features])
    model = sm.OLS(data[target], X).fit()
    predictions = model.predict(X)
    residuals = data[target] - predictions
    sse = np.sum(residuals ** 2)
    mse = mean_squared_error(data[target], predictions)
    r_squared = model.rsquared
    adjusted_r_squared = model.rsquared_adj
    n = len(data)
    k = len(features) + 1  # Number of predictors + intercept
    aic = n * np.log(sse / n) + 2 * k
    bic = n * np.log(sse / n) + k * np.log(n)
    mallows_cp = sse + 2 * k * mse - n * mse
    return {
        'Features': features,
        'AIC': aic,
        'BIC': bic,
        'R2': r_squared,
        'Adjusted R2': adjusted_r_squared,
        'MSE': mse,
        'SSE': sse,
        'Mallows Cp': mallows_cp
    }

# Predictor variables and target
predictors = ['RPM', 'Load', 'Coolant', 'Ambient', 'Throttle', 'Pressure']
target = 'Fuel'

# All combinations of predictors
combinations = []
for r in range(1, len(predictors) + 1):
    combinations.extend(itertools.combinations(predictors, r))

# Fit models and compute metrics
results = []
for combo in combinations:
    results.append(fit_model_and_calculate_metrics(list(combo), target, data.copy()))

# Results as DataFrame
results_df = pd.DataFrame(results)

# Print results sorted by AIC
print("Results sorted by AIC:")
print(results_df.sort_values(by='AIC'))

# Save results to a CSV file
results_df.to_csv("regression_results.csv", index=False)


Results sorted by AIC:
                                    Features        AIC        BIC        R2  \
36                [Load, Throttle, Pressure] -72.695266 -68.712337  0.998632   
46           [RPM, Load, Throttle, Pressure] -70.819024 -65.840363  0.998641   
53       [Load, Coolant, Throttle, Pressure] -70.748050 -65.769389  0.998636   
54       [Load, Ambient, Throttle, Pressure] -70.714040 -65.735378  0.998634   
58  [RPM, Load, Coolant, Throttle, Pressure] -68.882642 -62.908248  0.998645   
..                                       ...        ...        ...       ...   
18                       [Ambient, Throttle] -34.702470 -31.715274  0.989897   
37              [Coolant, Ambient, Throttle] -34.183502 -30.200573  0.990618   
3                                  [Ambient]  54.166934  56.158399  0.050158   
2                                  [Coolant]  54.276897  56.268362  0.044922   
15                        [Coolant, Ambient]  55.581386  58.568583  0.077564   

    Adjusted R2 